In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS caresync_adb.gold;

CREATE OR REPLACE TABLE caresync_adb.gold.patient_360 AS
WITH lab_results_agg AS(
    SELECT patient_id,
           CAST(COUNT(*) AS int) AS total_tests,
           CAST(SUM(CASE WHEN result_status = 'Abnormal' THEN 1 ELSE 0 END) AS int) AS abnormal_test_count,
           CAST(SUM(CASE WHEN result_status = 'Critical' THEN 1 ELSE 0 END) AS int) AS critical_test_count,
           MAX(test_date) AS last_test_date
    FROM caresync_adb.silver.lab_results
    GROUP BY patient_id
)
SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS date) AS date_of_birth,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name AS insurance_provider_name,
    COALESCE(lra.total_tests, 0) AS total_tests,
    COALESCE(lra.abnormal_test_count) AS abnormal_test_count,
    COALESCE(lra.critical_test_count) AS critical_test_count,
    lra.last_test_date
FROM caresync_adb.silver.patients p
LEFT JOIN caresync_adb.silver.hospitals h
    ON p.registered_hospital_id = h.hospital_id
LEFT JOIN caresync_adb.silver.insurance_providers ip
    ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_results_agg lra
    ON p.patient_id = lra.patient_id;

In [0]:
updated_df = spark.read.table('caresync_adb.gold.patient_360')
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))